# RAG Metrics 详解 · Lesson 2

## 课程目标

学完本课，你能做到：

- **读懂每个指标的数学定义**——不再靠直觉猜分数高低
- **手写 Python 实现 Retriever + Generator 指标**——不依赖框架黑盒
- **理解 LLM-as-Judge 的评估逻辑**——包括 Faithfulness 的 Claim 提取与 NLI 验证
- **生成高质量 Ground Truth 数据集**——解决 Recall 评估中最难的问题
- **掌握系统化诊断流程**——指标低了不慌，有一套排查决策树

## 快速回顾：RAG 核心指标

| 指标 | 评估对象 | 输入 | 分值范围 |
|------|---------|------|---------|
| **Context Precision** | 检索质量·精度 | retrieved chunks + ground truth | [0, 1] |
| **Context Recall** | 检索质量·覆盖 | retrieved chunks + ground truth | [0, 1] |
| **HitRate@k / MRR / NDCG** | 检索排序质量 | ranked list + relevance labels | [0, 1] |
| **Faithfulness** | 生成忠实度 | answer + retrieved context | [0, 1] |
| **Answer Relevancy** | 回答相关性 | question + answer | [0, 1] |

> 前三个指标衡量 **Retriever**；后两个指标衡量 **Generator**。优化方向不同，排查时必须分开看。

## 本节架构

```
Chapter 1  Context Precision  ──  公式推导 · 示例 · 阈值 · 代码
Chapter 2  Context Recall     ──  公式推导 · Ground Truth · 代码
Chapter 3  Ranking Metrics    ──  HitRate@k · P/R@k · MRR · NDCG · 代码
Chapter 4  Faithfulness       ──  Claim提取 · NLI · LLM-Judge · 代码
Chapter 5  Answer Relevancy   ──  反向问题 · Cosine · 代码
Chapter 6  诊断流程            ──  四象限 · 决策树 · 6种修复路径
Chapter 7  生产化              ──  A/B · 数据集 · RAGAS · 持续评估
```

**每章结构**：理论 → 公式 → 代码 → 踩坑

# RAG 评估指标 · 01 — Context Precision & Context Recall

**本 Notebook 目标：** 手动计算上下文精确率（Precision）与召回率（Recall），并使用 LLM 自动生成 Ground Truth。

## 学习目标

- 理解 Context Precision / Recall 的数学定义
- 用关键词重叠模拟检索器，从零实现指标计算
- 对比手工标注 Ground Truth 与 LLM 自动生成 Ground Truth

```bash
pip install openai python-dotenv
```

## 双指标一览

同一交集 **hit = |R ∩ GT|**，Precision 与 Recall 只是分母不同：

```mermaid
%%{init: {'flowchart': {'curve': 'linear', 'rankSpacing': 50, 'nodeSpacing': 30}}}%%
flowchart TB
    Q[Query] --> RET[Retrieved R]
    GT[Ground Truth GT] --> HIT[hit = R ∩ GT]
    RET --> HIT
    HIT --> P[Precision = hit / R]
    HIT --> RC[Recall = hit / GT]
```

- **Precision**：检索结果里有多少是相关的（噪声比例）
- **Recall**：该找的相关文档找全了多少（漏召回比例）

---
# Chapter 1 · Context Precision 深入

> 检索的噪声有多少？

## Precision 公式推导

**问题**：在检索到的 K 个 chunk 里，有多少是真正相关的？

| 符号 | 含义 |
|------|------|
| `K` | 检索返回的 chunk 总数（Top-K） |
| `C_i` | 第 `i` 个 chunk（`i = 1..K`） |
| `rel(C_i)` | 二值函数：chunk 相关返回 1，不相关返回 0 |
| `TP` | True Positives：相关 chunk 的数量 |

**公式**

$$\text{Context Precision} = \frac{\sum_{i=1}^{K} rel(C_i)}{K} = \frac{TP}{K}$$

**直觉**：Precision 回答"检索结果里有多少废料"。废料越多，Generator 受到的干扰越大，幻觉风险越高。

## Precision 计算流程

```mermaid
%%{init: {'flowchart': {'curve': 'linear', 'rankSpacing': 45}}}%%
flowchart TB
    A[Top-K chunks] --> B{Ci 在 GT 中?}
    B -->|是| C1[rel=1]
    B -->|否| C0[rel=0]
    C1 --> S[TP = sum rel]
    C0 --> S
    S --> F[Precision = TP / K]
```

## Precision 计算示例

**场景**：问题"什么是 Transformer？"，检索 Top-5

| 排名 | Chunk 内容摘要 | 相关？ | `rel(C_i)` |
|------|--------------|--------|-----------|
| 1 | Transformer 架构介绍 | 是 | 1 |
| 2 | BERT 预训练方法 | 是 | 1 |
| 3 | 卷积神经网络对比 | **否** | 0 |
| 4 | Attention 机制详解 | 是 | 1 |
| 5 | RNN 与梯度消失 | **否** | 0 |

$$\text{Precision} = \frac{1 + 1 + 0 + 1 + 0}{5} = \frac{3}{5} = 0.60$$

**Top-5 直观对照**（用表格代替复杂图，避免渲染重叠）：

| #1 | #2 | #3 | #4 | #5 |
|:--:|:--:|:--:|:--:|:--:|
| ✓ 相关 | ✓ 相关 | ✗ 噪声 | ✓ 相关 | ✗ 噪声 |

→ TP = 3，噪声 = 2，**Precision = 0.60**

## Binary vs. Weighted Precision

**Binary Precision（标准版）**——每个 chunk 只有"相关/不相关"两种状态：

```python
# rel(C_i) ∈ {0, 1}
precision = sum(rel) / K
```

优点：简单，计算快。缺点：忽略排名位置，第 1 名和第 5 名权重相同。

**Weighted Precision（加权版，RAGAS 默认）**——根据排名位置加权：

$$P@K_{weighted} = \frac{\sum_{i=1}^{K} P@i \cdot rel(C_i)}{\sum_{i=1}^{K} rel(C_i)}$$

其中 `P@i` = 前 i 名中相关 chunk 的比例。优点：奖励把相关 chunk 排在前面的检索器；缺点：实现复杂。

> RAGAS 用的是 Weighted 版本，你手动用 Binary 版对齐时会有 10-15% 偏差。本 Notebook 使用 **Binary 版**（doc_id 集合匹配）。

| 版本 | 计算方式 | 本课 / RAGAS |
|------|---------|-------------|
| Binary | TP / K | **本 Notebook** |
| Weighted | 按排名加权 | RAGAS 默认 |

## Precision 阈值选择：0.6 vs 0.8

| 阈值 | 含义 | 适合场景 | 风险 |
|------|------|---------|------|
| **0.6** | 每 5 个 chunk 允许 2 个噪声 | 知识库覆盖广、召回优先 | 噪声多，Generator 可能被干扰 |
| **0.8** | 每 5 个 chunk 最多 1 个噪声 | 精度优先、医疗/法律场景 | 可能漏掉相关文档，Recall 下降 |

**实际建议**

- 通用 QA 系统：起点设 **0.6**，迭代时逐步提高
- 高风险场景（医疗、法律、金融）：起点设 **0.75**，目标 **0.85+**
- Precision 和 Recall 是对立的——提高 Precision 通常会降低 Recall
- **不要孤立看 Precision**，必须和 Recall 一起看

> 经验法则：**Precision < 0.5 说明检索器严重失调，必须立即修**

## Precision ↔ Recall 此消彼长

```mermaid
%%{init: {'flowchart': {'curve': 'linear', 'rankSpacing': 45}}}%%
flowchart TB
    K[增大 Top-K] --> RU[Recall 上升]
    K --> PD[Precision 下降]
    RR[加 Reranker / 减小 K] --> PU[Precision 上升]
    RR --> RD[Recall 下降]
```

---
# Part 1 · 环境准备

In [2]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

---
# Part 2 · 知识库与评估函数（代码实战）

| 指标 | 公式 | 含义 |
|------|------|------|
| **Context Precision** | 相关检索数 / 检索总数 | 检索结果有多「干净」 |
| **Context Recall** | 检索到的相关数 / 全部相关数 | 相关文档有没有被「找全」 |

## 代码实现流程

```mermaid
%%{init: {'flowchart': {'curve': 'linear', 'rankSpacing': 45}}}%%
flowchart TB
    KB[KNOWLEDGE_BASE] --> RET[retrieve_by_keyword]
    RET --> RID[retrieved_ids]
    GT[relevant_ids] --> CP[compute_precision]
    RID --> CP
    GT --> CR[compute_recall]
    RID --> CR
```

In [3]:
# 模拟知识库（8 篇文档，涵盖 RAG/LLM 相关主题）
KNOWLEDGE_BASE = {
    "doc1": "RAG（检索增强生成）是一种将检索系统与大语言模型结合的技术，用于提升生成质量。",
    "doc2": "向量数据库（如 Chroma、Pinecone）用于存储文档嵌入，支持高效的相似度检索。",
    "doc3": "Chunk Size 是 RAG 中重要的超参数，过大导致噪声增多，过小导致语义不完整。",
    "doc4": "Reranker 模型可以对初步检索结果重新排序，显著提升 Context Precision。",
    "doc5": "LLM 幻觉（Hallucination）是指模型生成了与上下文不一致或凭空捏造的内容。",
    "doc6": "HyDE（假设文档嵌入）是一种 Query 增强技术，先让 LLM 生成假设答案再检索。",
    "doc7": "BM25 是经典的关键词检索算法，与向量检索结合可实现混合检索（Hybrid Search）。",
    "doc8": "RAGAS 是一个开源 RAG 评估框架，提供 Faithfulness、Answer Relevancy 等多个指标。",
}


def compute_precision(retrieved_ids: list, relevant_ids: set) -> float:
    """Context Precision = 检索结果中相关文档数 / 检索结果总数"""
    if not retrieved_ids:
        return 0.0
    relevant_retrieved = [doc_id for doc_id in retrieved_ids if doc_id in relevant_ids]
    return len(relevant_retrieved) / len(retrieved_ids)


def compute_recall(retrieved_ids: list, relevant_ids: set) -> float:
    """Context Recall = 检索到的相关文档数 / 全部相关文档数"""
    if not relevant_ids:
        return 0.0
    relevant_retrieved = [doc_id for doc_id in retrieved_ids if doc_id in relevant_ids]
    return len(relevant_retrieved) / len(relevant_ids)


def retrieve_by_keyword(query: str, top_k: int = 3) -> list:
    """基于关键词重叠度，从知识库中检索 top_k 篇文档（模拟 RAG 检索器）"""
    scores = {}
    query_words = set(query)
    for doc_id, content in KNOWLEDGE_BASE.items():
        overlap = sum(1 for char in query_words if char in content)
        scores[doc_id] = overlap
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [doc_id for doc_id, _ in ranked[:top_k]]


TEST_QUERIES = [
    {
        "query": "RAG 系统如何通过检索提升生成质量？",
        "relevant_ids": {"doc1", "doc2", "doc7"},
    },
    {
        "query": "如何减少 LLM 幻觉问题？",
        "relevant_ids": {"doc5", "doc1"},
    },
    {
        "query": "Reranker 和 Chunk Size 对检索的影响",
        "relevant_ids": {"doc3", "doc4"},
    },
]

---
# Part 3 · Precision & Recall 评估矩阵

对 3 条测试查询运行关键词检索，与手工标注的 Ground Truth 对比打分。

### 单条 Query 打分示意（top_k=3）

| 集合 | 内容 | 大小 |
|------|------|:--:|
| GT | doc1, doc2, doc7 | 3 |
| R（检索） | doc1, doc4, doc8 | 3 |
| **R ∩ GT** | doc1 | **1** |

```mermaid
%%{init: {'flowchart': {'curve': 'linear', 'rankSpacing': 45}}}%%
flowchart TB
    H[hit = 1] --> P[Precision = 1/3]
    H --> R[Recall = 1/3]
```

- doc4、doc8 → **噪声**（拉低 Precision）
- doc2、doc7 → **漏召回**（拉低 Recall）

In [4]:
print("=" * 60)
print("Context Precision & Recall 评估矩阵")
print("=" * 60)
print(f"{'查询':<30} {'Precision':>10} {'Recall':>10}")
print("-" * 60)

for item in TEST_QUERIES:
    query = item["query"]
    relevant_ids = item["relevant_ids"]
    retrieved_ids = retrieve_by_keyword(query, top_k=3)

    precision = compute_precision(retrieved_ids, relevant_ids)
    recall = compute_recall(retrieved_ids, relevant_ids)

    print(f"{query[:28]:<30} {precision:>10.2f} {recall:>10.2f}")
    print(f"  检索到: {retrieved_ids}")
    print(f"  相关集: {relevant_ids}")
    print()

print("=" * 60)

Context Precision & Recall 评估矩阵
查询                              Precision     Recall
------------------------------------------------------------
RAG 系统如何通过检索提升生成质量？                  0.67       0.67
  检索到: ['doc1', 'doc4', 'doc2']
  相关集: {'doc7', 'doc2', 'doc1'}

如何减少 LLM 幻觉问题？                       0.33       0.50
  检索到: ['doc5', 'doc6', 'doc2']
  相关集: {'doc5', 'doc1'}

Reranker 和 Chunk Size 对检索的影响         0.67       1.00
  检索到: ['doc3', 'doc4', 'doc2']
  相关集: {'doc4', 'doc3'}



---
# Chapter 2 · Context Recall 深入

> 所有该找到的信息，你找到了多少？

## Recall 公式推导

**问题**：Ground Truth 中包含的所有相关信息，检索器覆盖了几成？

| 符号 | 含义 |
|------|------|
| `GT` | Ground Truth：回答问题所需的全部相关 chunk 集合 |
| `R` | Retrieved：检索器实际返回的 chunk 集合 |
| `|GT ∩ R|` | GT 和 R 的交集大小（被找到的相关 chunk 数） |
| `|GT|` | Ground Truth 集合大小（应该找到的总数） |

**公式**

$$\text{Context Recall} = \frac{|GT \cap R|}{|GT|}$$

**直觉**：Recall 回答"有没有漏网之鱼"。Recall 低意味着 Generator 拿不到回答问题所需的关键信息，必然产生不完整或错误的回答。

## Recall 计算流程

```mermaid
%%{init: {'flowchart': {'curve': 'linear', 'rankSpacing': 45}}}%%
flowchart TB
    A[R 与 GT] --> B[求交集 hit]
    B --> C[Recall = hit / GT]
```

## 集合视角

```mermaid
%%{init: {'flowchart': {'curve': 'linear', 'rankSpacing': 45}}}%%
flowchart TB
    GT[GT 应找到 3 条] --> HIT[已找到 2 条]
    GT --> MISS[漏召回 1 条]
    HIT --> REC[Recall = 2/3]
```

**补充**：R 中多出来的噪声 chunk（在 R 但不在 GT）只影响 **Precision**，不影响 Recall 分母。

## Ground Truth 是什么？

**定义**：对于每个问题 Q，Ground Truth 是"**人类专家认为足以完整回答 Q 的最小 chunk 集合**"。

**为什么难生成？**

1. **主观性强**：不同专家对"相关"的判断不一致，边界模糊
2. **标注成本高**：100 条问题 × 平均 3 个 GT chunk = 300 次人工判断
3. **知识库变化**：知识库更新后 Ground Truth 可能失效
4. **粒度问题**：chunk_size 不同，同一信息可能分布在不同 chunk 里

| 策略 | 优点 | 缺点 |
|------|------|------|
| 人工标注 | 质量最高，可信 | 成本高，慢 |
| LLM 自动生成 | 快速，可扩展 | 可能有偏差，需人工抽样验证 |

> 推荐：**LLM 生成初稿 + 人工抽样 20% 验证**，兼顾效率和质量

## Ground Truth 生成流程

```mermaid
%%{init: {'flowchart': {'curve': 'linear', 'rankSpacing': 45}}}%%
flowchart TB
    Q[Query] --> MAN[人工标注]
    Q --> LLM[LLM 自动标注]
    MAN --> GT[relevant_ids]
    LLM --> GT
    GT --> EVAL[compute P / R]
```

## Ground Truth 生成策略 1：人工标注

**标准操作流程（SOP）**

1. **准备标注界面**：显示问题 Q + 知识库中所有 chunk 的摘要，标注员勾选"回答这个问题必须用到的 chunk"
2. **制定标注指南**：
   - 如果没有这个 chunk，回答会明显不完整或错误吗？→ 是 → 加入 GT
   - 这个 chunk 提供了额外上下文但非必需吗？→ 视情况
   - 这个 chunk 和问题完全无关吗？→ 不加入 GT
3. **双人独立标注 + 一致性校验**：Cohen's Kappa ≥ 0.7 才算合格
4. **构建评估数据集**：格式 `{"question": Q, "ground_truth_chunks": [C1, C2, ...]}`，建议每个 query 对应 2-5 个 GT chunk

## Ground Truth 生成策略 2：LLM 自动生成

**核心思路**：给 LLM 提供知识库内容，让它生成"应该检索到哪些 chunk"的判断。

**Prompt 模板要点**

- 给定问题和候选 Chunk 列表
- 返回 JSON：`required_indices` + `reasoning`
- 使用 `temperature=0.1` 确保稳定性

下方 **BONUS 单元**演示了 DeepSeek 自动生成 Ground Truth 的完整流程。

---
# BONUS · LLM 自动生成 Ground Truth

调用 DeepSeek 判断哪些文档与新问题相关，替代手工标注的 `relevant_ids`，再计算 Precision / Recall。

In [6]:
print("\n===== BONUS: LLM 自动生成 Ground Truth =====\n")

api_key = "sk-your-api-key-here"

client = OpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com",
)

new_query = "什么技术可以改善 RAG 的检索质量？"

docs_text = "\n".join(
    f"[{doc_id}] {content}" for doc_id, content in KNOWLEDGE_BASE.items()
)

prompt = f"""你是一位 RAG 评估专家。请判断以下文档中哪些与给定问题相关。

问题：{new_query}

文档列表：
{docs_text}

请返回相关文档的 ID 列表，格式如下（仅返回 JSON 数组，不要有其他文字）：
["doc1", "doc3", ...]"""

print(f"新查询：{new_query}")
print("正在调用 DeepSeek 生成 Ground Truth...\n")

try:
    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    raw = response.choices[0].message.content.strip()
    print(f"LLM 返回的相关文档 ID：{raw}")

    llm_relevant = set(json.loads(raw))
    retrieved = retrieve_by_keyword(new_query, top_k=3)

    p = compute_precision(retrieved, llm_relevant)
    r = compute_recall(retrieved, llm_relevant)
    print(f"\n基于 LLM Ground Truth 评估结果：")
    print(f"  检索到：{retrieved}")
    print(f"  LLM 相关集：{llm_relevant}")
    print(f"  Precision: {p:.2f}  |  Recall: {r:.2f}")
except Exception as e:
    print(f"LLM 调用出错：{e}")


===== BONUS: LLM 自动生成 Ground Truth =====

新查询：什么技术可以改善 RAG 的检索质量？
正在调用 DeepSeek 生成 Ground Truth...

LLM 返回的相关文档 ID：["doc2", "doc3", "doc4", "doc6", "doc7"]

基于 LLM Ground Truth 评估结果：
  检索到：['doc1', 'doc4', 'doc7']
  LLM 相关集：{'doc2', 'doc7', 'doc6', 'doc3', 'doc4'}
  Precision: 0.67  |  Recall: 0.40
